# 00 · Quickstart — Au end-to-end on the bundled NF-HEDM example

This notebook walks you through running the **full** NF-HEDM pipeline on the
Au example shipped at `NF_HEDM/Example/sim/`. It uses single-resolution mode
(no `GridRefactor` key in the param file → `NumLoops=0`).

What you'll see:

1. Inspect the bundled parameter file.
2. Run `midas_nf_pipeline.run_layer_pipeline` (one Python call → all stages).
3. Open the consolidated HDF5 and plot the orientation map.

The whole run is pure Python — no shelling out to C binaries.

In [ ]:
import os, shutil, tempfile, time
from pathlib import Path
from argparse import Namespace

import h5py
import numpy as np
import matplotlib.pyplot as plt

from midas_nf_pipeline.workflows import run_layer_pipeline
from midas_nf_pipeline.params import parse_parameters

## 1. Locate the Au example and copy it to a workspace

In [ ]:
MIDAS_HOME = Path(os.environ.get('MIDAS_HOME') or os.environ.get('MIDAS_INSTALL_DIR') or '.')
AU_DIR = MIDAS_HOME / 'NF_HEDM/Example/sim'
assert AU_DIR.exists(), f'Au example not found at {AU_DIR}'
print(list(AU_DIR.glob('test_*.txt')))

In [ ]:
# Sandbox: copy the param file (single-resolution; remove any GridRefactor)
ws = Path(tempfile.mkdtemp(prefix='nf_pipeline_quickstart_'))
src_param = AU_DIR / 'test_ps_au.txt'
dst_param = ws / src_param.name
text = src_param.read_text()
lines = [l for l in text.splitlines() if not l.strip().startswith('GridRefactor')]
lines.append(f'OutputDirectory {ws}')
dst_param.write_text('\n'.join(lines) + '\n')
print('workspace:', ws)
print('param  :', dst_param)

## 2. Run the pipeline (one call)

In [ ]:
args = Namespace(
    paramFN=str(dst_param),
    nCPUs=4,
    device='auto',
    ffSeedOrientations=False,
    doImageProcessing=1,
    startLayerNr=1, endLayerNr=1,
    resultFolder=str(ws),
    minConfidence=0.6,
    resume='', restartFrom='',
)
t0 = time.time()
h5_path = run_layer_pipeline(args, install_dir=str(MIDAS_HOME))
print(f'Done in {time.time()-t0:.1f}s. H5 → {h5_path}')

## 3. Inspect the consolidated HDF5

In [ ]:
with h5py.File(h5_path, 'r') as h5:
    def walk(name, obj):
        kind = 'GROUP' if hasattr(obj, 'keys') else 'DSET '
        if hasattr(obj, 'shape'):
            print(f'{kind} {name}  shape={obj.shape} dtype={obj.dtype}')
        else:
            print(f'{kind} {name}')
    h5.visititems(walk)

## 4. Plot the orientation map and KAM

In [ ]:
with h5py.File(h5_path, 'r') as h5:
    om = np.asarray(h5['maps/orientation'])  # (H, W, 7) — 7 fields
    extent = np.asarray(h5['maps/extent'])
    kam = np.asarray(h5['maps/kam']) if 'maps/kam' in h5 else None
    grain_id = np.asarray(h5['maps/grain_id']) if 'maps/grain_id' in h5 else None

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
im = axes[0].imshow(om[..., 0], origin='lower', cmap='viridis')
axes[0].set_title('Confidence')
plt.colorbar(im, ax=axes[0], shrink=0.8)
if kam is not None:
    im = axes[1].imshow(np.degrees(kam), origin='lower', cmap='magma')
    axes[1].set_title('KAM (deg)')
    plt.colorbar(im, ax=axes[1], shrink=0.8)
if grain_id is not None:
    im = axes[2].imshow(grain_id, origin='lower', cmap='tab20')
    axes[2].set_title('Grain ID')
    plt.colorbar(im, ax=axes[2], shrink=0.8)
plt.tight_layout()
plt.savefig(ws / 'au_summary.png', dpi=150)
print(f'plot saved to {ws / "au_summary.png"}')

## What just happened

`run_layer_pipeline` invoked the following stages, all in pure Python:

| Stage | Python module |
| --- | --- |
| Generate `hkls.csv` | `midas_hkls.write_nf_hkls_csv` |
| Seed orientations | `midas_nf_preprocess.seed_orientations` |
| Hex grid | `midas_nf_preprocess.hex_grid` |
| Tomo / mask filter | `midas_nf_preprocess.tomo_filter` |
| Diffraction spots | `midas_nf_preprocess.diffr_spots` |
| Image processing | `midas_nf_preprocess.process_images` |
| Orientation fitting | `midas_nf_fitorientation.fit_orientation_run` |
| ParseMic | `midas_nf_pipeline.parse_mic` |
| Consolidate H5 | `midas_nf_pipeline.consolidate` |

Crystal symmetry / misorientation come from `midas_stress.orientation` everywhere.